Bismillah
Startiong Project on:
Saturday, 26th Rabi-ul-Awwal, 1447 - 20th October, 2025

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split# , StratifiedKFold, cross_val_score

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier #, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.naive_bayes import MultinomialNB
# from sklearn.svm import SVC
# from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
# from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score #classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve

# from imblearn.over_sampling import SMOTE, RandomOverSampler
# from imblearn.under_sampling import RandomUnderSampler

import re
import string
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# For advanced preprocessing


# from textblob import TextBlob

# import matplotlib.pyplot as plt
# import seaborn as sns

# Download NLTK resources



In [ ]:
import nltk
try:
    nltk.download('stopwords', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
    nltk.download('averaged_perceptron_tagger', quiet=True)
except:
    print("Warning: Some NLTK downloads failed. Some features may not work.")

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

In [ ]:
# ============================================================================
# 1. LOAD YOUR DATA
# ============================================================================
# Replace 'your_dataset.csv' with your actual file path
df = pd.read_csv('Data/combined_dataset.csv')

print("Dataset shape:", df.shape)
print("Class distribution:\n", df['label'].value_counts())
print(f"Class imbalance ratio: {df['label'].value_counts()[0] / df['label'].value_counts()[1]:.2f}:1")
print("\nSample rows:\n", df.head(10))

In [ ]:
# ============================================================================
# 2. FEATURE ENGINEERING
# ============================================================================

def extract_text_features(text):
    """Extract handcrafted features from text"""
    features = {}
    
    # Basic length features
    features['text_length'] = len(text)
    features['word_count'] = len(text.split())
    features['avg_word_length'] = np.mean([len(word) for word in text.split()]) if text.split() else 0
    
    # Special character counts
    features['exclamation_count'] = text.count('!')
    features['question_count'] = text.count('?')
    features['uppercase_count'] = sum(1 for c in text if c.isupper())
    features['uppercase_ratio'] = features['uppercase_count'] / len(text) if len(text) > 0 else 0
    features['special_char_count'] = sum(1 for c in text if c in string.punctuation)
    features['digit_count'] = sum(1 for c in text if c.isdigit())
    
    # Profanity indicators (common toxic words)
    profanity_list = ['hate', 'stupid', 'idiot', 'kill', 'die', 'damn', 'hell', 'shut', 'fuck', 'shit', 'ass', 'bitch']
    features['profanity_count'] = sum(1 for word in text.lower().split() if word in profanity_list)
    
    # Sentiment analysis
    # try:
    #     try:
    #         blob = TextBlob(str(text))  # Ensure the input is a string
    #         sentiment = blob.sentiment
    #         features['sentiment_polarity'] = sentiment.polarity
    #         features['sentiment_subjectivity'] = sentiment.subjectivity
    #     except Exception as e:
    #         print(f"Error processing text: {text}. Error: {e}")
    #         features['sentiment_polarity'] = 0
    #         features['sentiment_subjectivity'] = 0
    # except:
    #     features['sentiment_polarity'] = 0
    #     features['sentiment_subjectivity'] = 0
    
    return features

print("\nExtracting text features...")
text_features = df['text'].apply(extract_text_features).apply(pd.Series)
print("Text features extracted:")
print(text_features.head())

In [ ]:
# ============================================================================
# 3. PREPROCESSING FUNCTIONS
# ============================================================================

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_minimal(text):
    """Minimal preprocessing - just lowercase"""
    return text.lower().strip()

def preprocess_basic(text):
    """Basic preprocessing - lowercase, remove extra spaces"""
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def preprocess_moderate(text):
    """Moderate preprocessing - remove URLs, mentions, hashtags"""
    text = text.lower().strip()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

def preprocess_aggressive(text):
    """Aggressive preprocessing - remove punctuation, numbers, special chars"""
    text = text.lower().strip()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[0-9]+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text)
    return text

# could not run the below strategies
def preprocess_with_stopwords(text):
    """Preprocessing with stopword removal"""
    text = preprocess_moderate(text)
    words = text.split()
    text = ' '.join([w for w in words if w not in stop_words])
    return text

def preprocess_with_stemming(text):
    """Preprocessing with stemming"""
    text = preprocess_moderate(text)
    words = text.split()
    text = ' '.join([stemmer.stem(w) for w in words])
    return text

def preprocess_with_lemmatization(text):
    """Preprocessing with lemmatization"""
    text = preprocess_moderate(text)
    words = text.split()
    text = ' '.join([lemmatizer.lemmatize(w) for w in words])
    return text

In [ ]:
# ============================================================================
# 4. EXPERIMENT CONFIGURATION
# ============================================================================

preprocessing_techniques = {
    'minimal': preprocess_minimal,
    'basic': preprocess_basic,
    'moderate': preprocess_moderate,
    'aggressive': preprocess_aggressive,
    'with_stopwords': preprocess_with_stopwords,
    'with_stemming': preprocess_with_stemming,
    'with_lemmatization': preprocess_with_lemmatization,
}

vectorization_configs = {
    'tfidf_basic': {
        'type': 'tfidf',
        'params': {'max_features': 5000, 'ngram_range': (1, 1)}
    },
    'tfidf_bigram': {
        'type': 'tfidf',
        'params': {'max_features': 5000, 'ngram_range': (1, 2)}
    },
    'tfidf_trigram': {
        'type': 'tfidf',
        'params': {'max_features': 10000, 'ngram_range': (1, 3)}
    },
    'count_basic': {
        'type': 'count',
        'params': {'max_features': 5000, 'ngram_range': (1, 1)}
    },
    'count_bigram': {
        'type': 'count',
        'params': {'max_features': 5000, 'ngram_range': (1, 2)}
    },
}

# sampling_strategies = {
#     'none': None,
#     'random_oversample': RandomOverSampler(random_state=42),
#     'random_undersample': RandomUnderSampler(random_state=42),
#     'smote': SMOTE(random_state=42, k_neighbors=3),
# }

# Expanded model configurations
model_configs = {
    'logistic_regression_balanced': {
        'model': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        'name': 'Logistic Regression (Balanced)'
    },
    'logistic_regression_c01': {
        'model': LogisticRegression(max_iter=1000, C=0.1, class_weight='balanced', random_state=42),
        'name': 'Logistic Regression (C=0.1)'
    },
    'logistic_regression_c10': {
        'model': LogisticRegression(max_iter=1000, C=10, class_weight='balanced', random_state=42),
        'name': 'Logistic Regression (C=10)'
    },
    'random_forest_balanced': {
        'model': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
        'name': 'Random Forest (100 trees)'
    },
    'random_forest_200': {
        'model': RandomForestClassifier(n_estimators=200, class_weight='balanced', max_depth=20, random_state=42, n_jobs=-1),
        'name': 'Random Forest (200 trees)'
    },
    'naive_bayes': {
        'model': MultinomialNB(),
        'name': 'Naive Bayes'
    },
    'naive_bayes_alpha01': {
        'model': MultinomialNB(alpha=0.1),
        'name': 'Naive Bayes (alpha=0.1)'
    },
    # 'gradient_boosting': {
    #     'model': GradientBoostingClassifier(n_estimators=100, random_state=42),
    #     'name': 'Gradient Boosting'
    # },
    'xgboost': {
        'model': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', use_label_encoder=False),
        'name': 'XGBoost'
    },
    'xgboost_tuned': {
        'model': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss', use_label_encoder=False),
        'name': 'XGBoost (Tuned)'
    },
    # 'lightgbm': {
    #     'model': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    #     'name': 'LightGBM'
    # },
    # 'lightgbm_tuned': {
    #     'model': LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1),
    #     'name': 'LightGBM (Tuned)'
    # } #,
    # 'svm_linear': {
    #     'model': SVC(kernel='linear', class_weight='balanced', random_state=42, probability=True),
    #     'name': 'SVM (Linear)'
    # },
}

In [ ]:
# ============================================================================
# 5. FULL EXPERIMENT RUNNER WITH INCREMENTAL SAVING
# ============================================================================

# Initialize CSV file with headers
results_csv_path = 'toxicity_experiment_results.csv'
csv_columns = ['experiment_id', 'preprocessing', 'vectorization', 'sampling', 'model', 
               'accuracy', 'f1_score', 'precision', 'recall', 'f1_toxic', 'f1_non_toxic',
               'best_threshold', 'accuracy_tuned', 'f1_tuned', 'precision_tuned', 'recall_tuned',
               'duration_seconds', 'timestamp', 'error']

# Create empty CSV with headers
pd.DataFrame(columns=csv_columns).to_csv(results_csv_path, index=False)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], 
    df['label'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['label']
)

print("\nStarting experiments...")
print(f"Train set size: {len(X_train)}, Test set size: {len(X_test)}")
print(f"Train class distribution: {y_train.value_counts().to_dict()}")
print(f"Test class distribution: {y_test.value_counts().to_dict()}\n")

total_experiments = (len(preprocessing_techniques) * 
                    len(vectorization_configs) * 
                    # len(sampling_strategies) * 
                    len(model_configs))

print(f"Total experiments to run: {total_experiments}")
print(f"Results will be saved incrementally to: {results_csv_path}\n")

experiment_count = 0

for prep_name, prep_func in preprocessing_techniques.items():
    print(f"\n{'='*80}")
    print(f"PREPROCESSING: {prep_name}")
    print(f"{'='*80}")
    
    X_train_prep = X_train.apply(prep_func)
    X_test_prep = X_test.apply(prep_func)
    
    for vec_name, vec_config in vectorization_configs.items():
        print(f"\n  Vectorization: {vec_name}")
        
        if vec_config['type'] == 'tfidf':
            vectorizer = TfidfVectorizer(**vec_config['params'])
        else:
            vectorizer = CountVectorizer(**vec_config['params'])
        
        X_train_vec = vectorizer.fit_transform(X_train_prep)
        X_test_vec = vectorizer.transform(X_test_prep)
        
        # for samp_name, sampler in sampling_strategies.items():
            
        #     if sampler is not None:
        #         X_train_samp, y_train_samp = sampler.fit_resample(X_train_vec, y_train)
        #     else:
        #         X_train_samp, y_train_samp = X_train_vec, y_train
            
        #     for model_key, model_config in model_configs.items():
        #         experiment_count += 1
                
        #         try:
        #             model = model_config['model']
        #             start_time = datetime.now()
                    
        #             model.fit(X_train_samp, y_train_samp)
                    
        #             # Default predictions
        #             y_pred = model.predict(X_test_vec)
                    
        #             # Get probability predictions for threshold tuning
        #             if hasattr(model, 'predict_proba'):
        #                 y_proba = model.predict_proba(X_test_vec)[:, 1]
                        
        #                 # Find optimal threshold
        #                 thresholds = np.arange(0.1, 0.9, 0.05)
        #                 best_threshold = 0.5
        #                 best_f1_threshold = 0
                        
        #                 for thresh in thresholds:
        #                     y_pred_thresh = (y_proba >= thresh).astype(int)
        #                     f1_thresh = f1_score(y_test, y_pred_thresh)
        #                     if f1_thresh > best_f1_threshold:
        #                         best_f1_threshold = f1_thresh
        #                         best_threshold = thresh
                        
        #                 # Use optimal threshold
        #                 y_pred_tuned = (y_proba >= best_threshold).astype(int)
        #             else:
        #                 y_pred_tuned = y_pred
        #                 best_threshold = 0.5
        #                 y_proba = None
                    
        #             # Calculate metrics (default threshold)
        #             accuracy = accuracy_score(y_test, y_pred)
        #             f1 = f1_score(y_test, y_pred)
        #             precision = precision_score(y_test, y_pred)
        #             recall = recall_score(y_test, y_pred)
        #             f1_toxic = f1_score(y_test, y_pred, pos_label=1)
        #             f1_non_toxic = f1_score(y_test, y_pred, pos_label=0)
                    
        #             # Calculate metrics (tuned threshold)
        #             accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
        #             f1_tuned = f1_score(y_test, y_pred_tuned)
        #             precision_tuned = precision_score(y_test, y_pred_tuned)
        #             recall_tuned = recall_score(y_test, y_pred_tuned)
                    
        #             end_time = datetime.now()
        #             duration = (end_time - start_time).total_seconds()
                    
        #             # Store result
        #             result = {
        #                 'experiment_id': experiment_count,
        #                 'preprocessing': prep_name,
        #                 'vectorization': vec_name,
        #                 'sampling': samp_name,
        #                 'model': model_config['name'],
        #                 'accuracy': accuracy,
        #                 'f1_score': f1,
        #                 'precision': precision,
        #                 'recall': recall,
        #                 'f1_toxic': f1_toxic,
        #                 'f1_non_toxic': f1_non_toxic,
        #                 'best_threshold': best_threshold,
        #                 'accuracy_tuned': accuracy_tuned,
        #                 'f1_tuned': f1_tuned,
        #                 'precision_tuned': precision_tuned,
        #                 'recall_tuned': recall_tuned,
        #                 'duration_seconds': duration,
        #                 'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        #                 'error': None
        #             }
                    
        #             # Save to CSV immediately
        #             pd.DataFrame([result]).to_csv(results_csv_path, mode='a', header=False, index=False)
                    
        #             if experiment_count % 10 == 0:
        #                 print(f"    [{experiment_count}/{total_experiments}] Completed - F1: {f1_tuned:.4f}")
                
        #         except Exception as e:
        #             print(f"    ERROR with {model_config['name']}: {str(e)}")
        #             result = {
        #                 'experiment_id': experiment_count,
        #                 'preprocessing': prep_name,
        #                 'vectorization': vec_name,
        #                 'sampling': samp_name,
        #                 'model': model_config['name'],
        #                 'accuracy': None,
        #                 'f1_score': None,
        #                 'precision': None,
        #                 'recall': None,
        #                 'f1_toxic': None,
        #                 'f1_non_toxic': None,
        #                 'best_threshold': None,
        #                 'accuracy_tuned': None,
        #                 'f1_tuned': None,
        #                 'precision_tuned': None,
        #                 'recall_tuned': None,
        #                 'duration_seconds': None,
        #                 'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        #                 'error': str(e)
        #             }
                    
        #             # Save error to CSV immediately
        #             pd.DataFrame([result]).to_csv(results_csv_path, mode='a', header=False, index=False)

print(f"\n{'='*80}")
print("ALL EXPERIMENTS COMPLETED!")
print(f"{'='*80}\n")

In [ ]:
# ============================================================================
# 6. RESULTS ANALYSIS
# ============================================================================

# Load results from CSV
results_df = pd.read_csv(results_csv_path)

print(f"✓ Results loaded from '{results_csv_path}'")
print(f"Total experiments completed: {len(results_df)}")

# Display top performers by F1 score
print("\n" + "="*80)
print("TOP 15 MODELS BY F1 SCORE (TUNED THRESHOLD)")
print("="*80)
top_f1 = results_df.nlargest(15, 'f1_tuned')[['model', 'preprocessing', 'vectorization', 
                                                'sampling', 'f1_tuned', 'accuracy_tuned', 
                                                'precision_tuned', 'recall_tuned', 'best_threshold']]
print(top_f1.to_string(index=False))

# Display top performers by accuracy
print("\n" + "="*80)
print("TOP 15 MODELS BY ACCURACY")
print("="*80)
top_acc = results_df.nlargest(15, 'accuracy_tuned')[['model', 'preprocessing', 'vectorization', 
                                                       'sampling', 'accuracy_tuned', 'f1_tuned', 
                                                       'precision_tuned', 'recall_tuned']]
print(top_acc.to_string(index=False))

# Best balanced model
print("\n" + "="*80)
print("MOST BALANCED MODELS (Best F1 for toxic class)")
print("="*80)
top_balanced = results_df.nlargest(10, 'f1_toxic')[['model', 'preprocessing', 'vectorization', 
                                                     'sampling', 'f1_toxic', 'f1_non_toxic', 
                                                     'f1_score', 'accuracy']]
print(top_balanced.to_string(index=False))

# Summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Best F1 Score (default): {results_df['f1_score'].max():.4f}")
print(f"Best F1 Score (tuned): {results_df['f1_tuned'].max():.4f}")
print(f"Best Accuracy: {results_df['accuracy'].max():.4f}")
print(f"Average F1 Score: {results_df['f1_score'].mean():.4f}")
print(f"Average Accuracy: {results_df['accuracy'].mean():.4f}")

# Performance by preprocessing technique
print("\n" + "="*80)
print("AVERAGE PERFORMANCE BY PREPROCESSING TECHNIQUE")
print("="*80)
prep_summary = results_df.groupby('preprocessing')[['accuracy', 'f1_score', 'f1_tuned']].mean().sort_values('f1_tuned', ascending=False)
print(prep_summary)

# Performance by vectorization method
print("\n" + "="*80)
print("AVERAGE PERFORMANCE BY VECTORIZATION METHOD")
print("="*80)
vec_summary = results_df.groupby('vectorization')[['accuracy', 'f1_score', 'f1_tuned']].mean().sort_values('f1_tuned', ascending=False)
print(vec_summary)

# Performance by model type
print("\n" + "="*80)
print("AVERAGE PERFORMANCE BY MODEL TYPE")
print("="*80)
model_summary = results_df.groupby('model')[['accuracy', 'f1_score', 'f1_tuned']].mean().sort_values('f1_tuned', ascending=False)
print(model_summary)

# Performance by sampling strategy
print("\n" + "="*80)
print("AVERAGE PERFORMANCE BY SAMPLING STRATEGY")
print("="*80)
samp_summary = results_df.groupby('sampling')[['accuracy', 'f1_score', 'f1_tuned']].mean().sort_values('f1_tuned', ascending=False)
print(samp_summary)

# Check for errors
errors = results_df[results_df['error'].notna()]
if len(errors) > 0:
    print("\n" + "="*80)
    print(f"EXPERIMENTS WITH ERRORS: {len(errors)}")
    print("="*80)
    print(errors[['experiment_id', 'model', 'preprocessing', 'vectorization', 'sampling', 'error']])

# Threshold tuning impact
print("\n" + "="*80)
print("THRESHOLD TUNING IMPACT")
print("="*80)
results_df['f1_improvement'] = results_df['f1_tuned'] - results_df['f1_score']
avg_improvement = results_df['f1_improvement'].mean()
max_improvement = results_df['f1_improvement'].max()
print(f"Average F1 improvement from threshold tuning: {avg_improvement:.4f}")
print(f"Maximum F1 improvement from threshold tuning: {max_improvement:.4f}")

best_improvement = results_df.nlargest(10, 'f1_improvement')[['model', 'preprocessing', 'f1_score', 'f1_tuned', 'f1_improvement', 'best_threshold']]
print("\nTop 10 models with best improvement from threshold tuning:")
print(best_improvement.to_string(index=False))

print("\n" + "="*80)
print("EXPERIMENT COMPLETE!")
print("="*80)
print(f"\n✅ All results saved in: {results_csv_path}")